# Disentanglement Sweep: classifier_lambda & discriminator_lambda

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os, sys
sys.path.append('src')

import numpy as np
import scanpy as sc
import scvi

In [ ]:
import cellina
from cellina import CellinaModel
from src.cellina._spatial_utils import spatial_neighbors, compute_spatial_features
from perturb_utils import load_crc_slide

scvi.settings.seed = 420
print(cellina.__version__)

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
slide_id      = 242
labels_key    = 'coarse_type'
domains_key   = 'typ'
batch_size    = 512
lambda_values = [1e-5, 1, 50]  #[1e-5, 1e-3, 0.01, 0.1, 0.5, 1, 2, 5, 50]
save_dir      = 'results/disentanglement_sweep'
adata_path    = f'{save_dir}/sweep_results.h5ad'
os.makedirs(save_dir, exist_ok=True)

In [ ]:
## 1. Data
adata = load_crc_slide(slide_id, labels_key=labels_key, domains_key=domains_key)
spatial_neighbors(adata, bandwidth=100 / 0.12028, max_neighbours=200, standardize=False)
compute_spatial_features(adata)
print(adata)

In [ ]:
## 2. Sweep
for lam in lambda_values:
    print(f"\n=== lambda = {lam} ===")

    CellinaModel.setup_anndata(
        adata,
        batch_key=None,
        labels_key=labels_key,
        domains_key=domains_key,
        layer='counts',
        spatial_obsm_key='spatial_x',
    )

    model = CellinaModel(
        adata,
        n_latent=20,
        classifier_lambda=lam,
        discriminator_lambda=lam,
        n_layers=3,
        condition_on_intrinsic=False,
    )

    model.train(
        max_epochs=100,
        check_val_every_n_epoch=1,
        early_stopping=True,
        early_stopping_patience=5,
        early_stopping_monitor='vae_loss_validation',
        train_size=0.9,
        validation_size=0.1,
        plan_kwargs={'lr': 0.001, 'weight_decay': 0.0001, 'normalize_losses': True},
        enable_checkpointing=True,
        batch_size=batch_size,
        devices=[0],
    )
    
    print(f"{lam} Training history:")
    print(model.history_['classifier_loss_validation'].plot())
    print(model.history_['classifier_accuracy_validation'].plot())

    key = str(lam)
    adata.obsm[f'X_cellina_z_{key}'] = model.get_latent_representation(adata, latent_key='z')
    adata.obsm[f'X_cellina_s_{key}'] = model.get_latent_representation(adata, latent_key='s')

    adata.write_h5ad(adata_path)
    print(f"  Saved → {adata_path}")

In [ ]:
print(model.history_['classifier_loss_validation'].plot())

In [ ]:
print(model.history_['classifier_loss_raw_validation'].plot())

In [ ]:
print(model.history_['fool_loss_raw_validation'].plot())

## 3. Evaluation

In [ ]:
adata = sc.read_h5ad(adata_path)

In [ ]:
from disentanglement_eval import (
    benchmark_disentanglement, aggregate_results,
    min_max_scale_metrics, plot_results_table, WEIGHTS, _METRIC_TYPE,
)

adata_eval = sc.read_h5ad(adata_path)
# subset for faster evaluation
adata_eval = sc.pp.subsample(adata_eval, fraction=0.5, copy=True)

emb_keys = sorted([k for k in adata_eval.obsm if k.startswith('X_cellina')])
z_emb_keys = [f'X_cellina_z_{lam}' for lam in lambda_values]
s_emb_keys = [f'X_cellina_s_{lam}' for lam in lambda_values]
print(emb_keys)

In [ ]:
print(adata_eval.shape)

In [ ]:
# z: cell type (labels_key) is the biological signal; spatial domain (domains_key) is nuisance
individual_z = benchmark_disentanglement(
    adata=adata_eval,
    embedding_obsm_keys=z_emb_keys,
    nuisance_key=domains_key,    # 'typ' — spatial domain is nuisance for z
    bio_key=labels_key,           # 'coarse_type' — label conservation for z
    batch_key=None,
    n_jobs=24,
)
results_z = aggregate_results(individual_z, **WEIGHTS)


In [ ]:

# s: spatial domain (domains_key) is the biological signal; cell type (labels_key) is nuisance
individual_s = benchmark_disentanglement(
    adata=adata_eval,
    embedding_obsm_keys=s_emb_keys,
    nuisance_key=labels_key,     # 'coarse_type' — cell type is nuisance for s
    bio_key=domains_key,          # 'typ' — domain conservation for s
    batch_key=None,
    n_jobs=24,
)
results_s = aggregate_results(individual_s, **WEIGHTS)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.minor.size": 1.5,
    "xtick.minor.width": 0.4,
    "lines.linewidth": 1.2,
    "lines.markersize": 4,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "figure.dpi": 300,
})

# ── Extract actual data from benchmark results ─────────────────────────────────
params = np.array(lambda_values)

def extract_metric(results_df, emb_keys, col):
    return [results_df.loc[k, col] for k in emb_keys]

data = {
    "s": {
        "Domain conservation": extract_metric(results_s, s_emb_keys, "Bio conservation"),
        "Nuisance score":      extract_metric(results_s, s_emb_keys, "Nuisance score"),
        "Total":               extract_metric(results_s, s_emb_keys, "Total"),
    },
    "z": {
        "Label conservation":  extract_metric(results_z, z_emb_keys, "Bio conservation"),
        "Nuisance score":      extract_metric(results_z, z_emb_keys, "Nuisance score"),
        "Total":               extract_metric(results_z, z_emb_keys, "Total"),
    },
}

# ── Color palette (Nature Methods-compatible, Okabe-Ito) ──────────────────────
_colors = {
    "Total":                "#D55E00",
    "Label conservation":   "#0072B2",   # deep blue
    "Domain conservation":  "#56B4E9",   # sky blue
    "Nuisance score":       "#009E73",
}
_markers = {
    "Total":                "o",
    "Label conservation":   "s",
    "Domain conservation":  "s",
    "Nuisance score":       "^",
}

panels = [
    ("z", "Label conservation",  "z latent", "a"),
    ("s", "Domain conservation", "s latent", "b"),
]

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 2,
    figsize=(6, 2),
    constrained_layout=True,
)

for ax, (key, bio_label, title, letter) in zip(axes, panels):

    palette = {
        bio_label:        _colors[bio_label],
        "Nuisance score": _colors["Nuisance score"],
        "Total":          _colors["Total"],
    }
    panel_markers = {
        bio_label:        _markers[bio_label],
        "Nuisance score": _markers["Nuisance score"],
        "Total":          _markers["Total"],
    }

    for metric, color in palette.items():
        ax.plot(
            params,
            data[key][metric],
            color=color,
            marker=panel_markers[metric],
            markerfacecolor="white",
            markeredgecolor=color,
            markeredgewidth=0.8,
            label=metric,
            clip_on=False,
        )

    ax.set_xscale("log")
    ax.set_xlabel("Hyperparameter value")
    ax.set_title(title, pad=3)

    ax.set_xticks(params)
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f"$10^{{{int(round(np.log10(x)))}}}$" if x < 0.01 else f"{x:g}"
        )
    )
    ax.tick_params(axis="x", which="both", bottom=True)
    ax.xaxis.set_minor_locator(mticker.NullLocator())
    ax.spines[["top", "right"]].set_visible(False)

    ax.set_ylabel("Score")
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator(2))
    ax.tick_params(axis="y", which="minor", left=True)

    ax.text(
        -0.22, 1.08, letter,
        transform=ax.transAxes,
        fontsize=8, fontweight="bold", va="top",
    )

# ── Shared legend below both panels ───────────────────────────────────────────
# z-panel contributes: Label conservation, Nuisance score, Total
# s-panel contributes: Domain conservation (bio proxy), Nuisance score, Total
# Legend order: Label conservation | Domain conservation | Nuisance score | Total
handles_z, labels_z = axes[0].get_legend_handles_labels()  # [bio_z, nui, tot]
handles_s, labels_s = axes[1].get_legend_handles_labels()  # [bio_s, nui, tot]

legend_handles = [handles_z[0], handles_s[0], handles_z[1], handles_z[2]]
legend_labels  = [labels_z[0],  labels_s[0],  labels_z[1],  labels_z[2]]

fig.legend(
    legend_handles, legend_labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.28),
    ncol=4,
    frameon=False,
    handlelength=1.5,
    handletextpad=0.4,
    columnspacing=1.0,
)

fig.savefig(f"{save_dir}/cellina_scores.pdf", bbox_inches="tight", dpi=300)
fig.savefig(f"{save_dir}/cellina_scores.png", bbox_inches="tight", dpi=300)
print("Saved.")

## All Metrics

In [ ]:
results = benchmark_disentanglement(
    adata=adata_eval,
    embedding_obsm_keys=z_emb_keys + s_emb_keys,
    nuisance_key=domains_key,    # 'typ' — spatial domain is nuisance for z
    bio_key=labels_key,           # 'coarse_type' — label conservation for z
    batch_key=None,
    n_jobs=24,
)

results_agg = aggregate_results(results, **WEIGHTS)

In [ ]:
plot_results_table(results_agg, metric_type=_METRIC_TYPE["Bio conservation"], save_path=f"{save_dir}/disentanglement_sweep_bio_conservation.pdf")